# 03 — Download MWA data

Find a solar MWA observation and download it from the ASVO. Imaging of files already on disk is in `03_mwa_imaging.ipynb`.

`giant-squid` runs on the Mac. `$MWA_ASVO_API_KEY` must be set. A visibility job can sit in the queue for 10–60 minutes.

In [ ]:
import subprocess
import urllib.request
from pathlib import Path

import pandas as pd
import pyvo
from astropy.time import Time, TimeDelta

## 1. Find solar MWA observations via TAP

Query the [MWA VO TAP service](https://vo.mwatelescope.org/mwa_asvo/tap) for observations pointing near the sun (`sun_elevation > 20`).

In [ ]:
# Sharma et al. 2022 (ApJ 937, 99): seven ~5 min solar scans on 2015-12-03
event_start = Time('2015-12-03 03:24:31', scale='utc')
event_end   = Time('2015-12-03 03:54:31', scale='utc')

gps_start = int(event_start.gps)
gps_end   = int(event_end.gps)
proprietary = int((Time.now() - TimeDelta(548, format='jd')).gps)

tap = pyvo.dal.TAPService('https://vo.mwatelescope.org/mwa_asvo/tap')

obs = (
    tap.search(f"""
        SELECT TOP 20 *
        FROM mwa.observation
        WHERE obs_id BETWEEN {gps_start} AND {gps_end}
        AND sun_elevation > 20
        AND obs_id < {proprietary}
        AND good_tiles >= 112
        AND dataquality <= 1
        AND deleted_flag != 'TRUE'
        AND gpubox_files_archived > 1
        ORDER BY obs_id DESC
    """)
    .to_table()
    .to_pandas()
    .dropna(axis=1, how='all')
)

obs['config']    = obs['mwa_array_configuration'].str.split(' ').str[-1]
obs['gigabytes'] = obs['total_archived_data_bytes'] / 1e9

print(f"{len(obs)} observations found")
obs[['obs_id', 'starttime_utc', 'obsname', 'config', 'sun_elevation', 'gigabytes']]

## 2. Choose an observation

In [ ]:
# None uses the first TAP result. Set an obsid to download a specific observation.
OBSID = None
obsid = int(OBSID) if OBSID is not None else int(obs['obs_id'].iloc[0])
data_dir = Path('../data') / str(obsid)
(data_dir / 'raw').mkdir(parents=True, exist_ok=True)

print(f"obsid    : {obsid}")
print(f"data_dir : {data_dir.resolve()}")

## 3. Metafits

The metafits is public and small. Visibility files come from the ASVO job below.

In [ ]:
metafits_path = data_dir / f'{obsid}.metafits'
if metafits_path.exists():
    print(f'Already on disk: {metafits_path}')
else:
    urllib.request.urlretrieve(
        f'https://ws.mwatelescope.org/metadata/fits?obs_id={obsid}',
        metafits_path,
    )
    print(f'Wrote {metafits_path}')

## 4. Download visibilities with giant-squid

Submit the job, then re-run the status cell until it says Ready, then download.

In [ ]:
result = subprocess.run(
    ['giant-squid', 'submit-vis', str(obsid)],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)

In [ ]:
result = subprocess.run(
    ['giant-squid', 'list', str(obsid)],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)

In [ ]:
JOB_ID = None  # from the giant-squid list table, once the job is Ready

if JOB_ID:
    result = subprocess.run(
        ['giant-squid', 'download', str(JOB_ID), '--keep-zip', '--download-dir', str(data_dir / 'raw')],
        capture_output=True, text=True,
    )
    print(result.stdout)
    print(result.stderr)
else:
    print('Set JOB_ID from the list cell, then run this cell again.')